## Import

In [7]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, FloatType, IntegerType, StringType
from pyspark.sql.window import Window

StatementMeta(, dfa3a73b-3dd1-4f8e-b041-b9431255bf8f, 9, Finished, Available, Finished, False)

## Load Bronze Tables

In [2]:
df_sales    = spark.read.format("delta").table("bronze_sales")
df_stores   = spark.read.format("delta").table("bronze_stores")
df_oil      = spark.read.format("delta").table("bronze_oil")
df_holidays = spark.read.format("delta").table("bronze_holidays_events")
df_txns     = spark.read.format("delta").table("bronze_transactions")
 
for name, df in {"sales":df_sales,"stores":df_stores,"oil":df_oil,
                 "holidays":df_holidays,"txns":df_txns}.items():
    print(f"  bronze_{name:<15} → {df.count():>10,} rows")

StatementMeta(, dfa3a73b-3dd1-4f8e-b041-b9431255bf8f, 4, Finished, Available, Finished, False)

  bronze_sales           →  3,000,888 rows
  bronze_stores          →         54 rows
  bronze_oil             →      1,218 rows
  bronze_holidays        →        350 rows
  bronze_txns            →     83,488 rows


## Clean Sales

In [4]:
df_sales_clean = (
    df_sales
    .withColumn("date",        F.col("date").cast(DateType()))
    .withColumn("store_nbr",   F.col("store_nbr").cast(IntegerType()))
    .withColumn("sales",       F.col("sales").cast(FloatType()))
    .withColumn("onpromotion", F.col("onpromotion").cast(IntegerType()))
    .dropna(subset=["date", "store_nbr", "family", "sales"])
    .filter(F.col("sales") >= 0)
    .withColumn("year",         F.year("date"))
    .withColumn("month",        F.month("date"))
    .withColumn("day",          F.dayofmonth("date"))
    .withColumn("weekday",      F.dayofweek("date"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("quarter",      F.quarter("date"))
    # .withColumn("is_month_end",
    #     F.when(F.col("day") == F.last_day(F.col("date")).cast(DateType()), 1).otherwise(0))
    .withColumn("is_month_end", 
        F.when(F.col("date") == F.last_day(F.col("date")), 1).otherwise(0))
    .withColumn("is_month_start",
        F.when(F.col("day") == 1, 1).otherwise(0))
)
print(f"\nClean sales rows: {df_sales_clean.count():,}")

StatementMeta(, dfa3a73b-3dd1-4f8e-b041-b9431255bf8f, 6, Finished, Available, Finished, False)


Clean sales rows: 3,000,888


## Forward-fill Oil Prices (no gaps on weekends)

In [5]:
min_date = df_oil.agg(F.min(F.col("date").cast(DateType()))).collect()[0][0]
max_date = df_oil.agg(F.max(F.col("date").cast(DateType()))).collect()[0][0]
 
date_range = spark.sql(f"""
    SELECT explode(sequence(date('{min_date}'), date('{max_date}'),
                   interval 1 day)) AS date
""")
 
w_oil = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)
 
df_oil_clean = (
    date_range
    .join(df_oil.withColumn("date", F.col("date").cast(DateType()))
               .withColumn("dcoilwtico", F.col("dcoilwtico").cast(FloatType())),
          on="date", how="left")
    .withColumn("oil_price",
        F.last("dcoilwtico", ignorenulls=True).over(w_oil))
    .select("date", "oil_price")
    .fillna({"oil_price": 65.0})
)
print(f"Oil price nulls after fill: {df_oil_clean.filter(F.col('oil_price').isNull()).count()}")

StatementMeta(, dfa3a73b-3dd1-4f8e-b041-b9431255bf8f, 7, Finished, Available, Finished, False)

Oil price nulls after fill: 0


## Clean Holidays

In [8]:
df_holidays_clean = (
    df_holidays
    .withColumn("date", F.col("date").cast(DateType()))
    .filter(
        F.col("transferred").isNull() |
        (F.lower(F.col("transferred").cast(StringType())) == "false")
    )
    .withColumn("holiday_scope",
        F.when(F.col("locale") == "National", "national")
        .when(F.col("locale") == "Regional",  "regional")
        .otherwise("local"))
    .withColumn("is_holiday", F.lit(1))
    .select("date", "type", "holiday_scope", "locale_name",
            "description", "is_holiday")
    .distinct()
)
print(f"Holiday records: {df_holidays_clean.count()}")

StatementMeta(, dfa3a73b-3dd1-4f8e-b041-b9431255bf8f, 10, Finished, Available, Finished, False)

Holiday records: 338


## Clean Stores & Transactions

In [9]:
df_stores_clean = (
    df_stores
    .withColumn("store_nbr", F.col("store_nbr").cast(IntegerType()))
    .withColumn("cluster",   F.col("cluster").cast(IntegerType()))
    .withColumn("type", F.upper(F.trim(F.col("type"))))
    .dropna(subset=["store_nbr"])
)
 
df_txns_clean = (
    df_txns
    .withColumn("date",         F.col("date").cast(DateType()))
    .withColumn("store_nbr",    F.col("store_nbr").cast(IntegerType()))
    .withColumn("transactions", F.col("transactions").cast(IntegerType()))
    .dropna(subset=["date", "store_nbr", "transactions"])
    .filter(F.col("transactions") > 0)
)

StatementMeta(, dfa3a73b-3dd1-4f8e-b041-b9431255bf8f, 11, Finished, Available, Finished, False)

##  Master Join (Silver)

In [10]:
df_silver = (
    df_sales_clean
    .join(df_stores_clean.select("store_nbr","city","state","type","cluster"),
          on="store_nbr", how="left")
    .join(df_oil_clean, on="date", how="left")
    .join(df_txns_clean.select("date","store_nbr","transactions"),
          on=["date","store_nbr"], how="left")
    .join(df_holidays_clean
              .filter(F.col("holiday_scope") == "national")
              .select("date","is_holiday").distinct(),
          on="date", how="left")
    .fillna({"is_holiday":0, "oil_price":65.0, "transactions":0, "cluster":0})
    .withColumnRenamed("type", "store_type")
)
 
print(f"\nsilver_master_sales: {df_silver.count():,} rows")

StatementMeta(, dfa3a73b-3dd1-4f8e-b041-b9431255bf8f, 12, Finished, Available, Finished, False)


silver_master_sales: 3,000,888 rows


## Write Silver Tables

In [11]:
df_silver.write.format("delta").mode("overwrite") \
         .option("overwriteSchema","true").saveAsTable("silver_master_sales")
df_stores_clean.write.format("delta").mode("overwrite").saveAsTable("silver_stores")
df_oil_clean.write.format("delta").mode("overwrite").saveAsTable("silver_oil")
df_holidays_clean.write.format("delta").mode("overwrite").saveAsTable("silver_holidays")
 
print("\n All Silver tables written. Proceed to 03_feature_engineering.py")

StatementMeta(, dfa3a73b-3dd1-4f8e-b041-b9431255bf8f, 13, Finished, Available, Finished, False)


 All Silver tables written. Proceed to 03_feature_engineering.py


## Data Quality Assertion Report

In [12]:
df_chk  = spark.read.format("delta").table("silver_master_sales")
total   = df_chk.count()
crit    = ["sales","store_nbr","family","date","city","oil_price","is_holiday"]
print("\nData Quality Report:")
for col in crit:
    n   = df_chk.filter(F.col(col).isNull()).count()
    pct = n/total*100
    print(f"  {'✅' if pct==0 else '⚠️ '} {col:<20} nulls: {n:>5,}  ({pct:.1f}%)")

StatementMeta(, dfa3a73b-3dd1-4f8e-b041-b9431255bf8f, 14, Finished, Available, Finished, False)


Data Quality Report:
  ✅ sales                nulls:     0  (0.0%)
  ✅ store_nbr            nulls:     0  (0.0%)
  ✅ family               nulls:     0  (0.0%)
  ✅ date                 nulls:     0  (0.0%)
  ✅ city                 nulls:     0  (0.0%)
  ✅ oil_price            nulls:     0  (0.0%)
  ✅ is_holiday           nulls:     0  (0.0%)
